In [44]:
from common import *
from db import get_engine

engine = get_engine()

## 1. Ümumi KPI nəticələrinin hazırlanması

Bu mərhələdə Silver cədvəlindən əsas uçuş KPI-ları hesablanır. Məqsəd Power BI və reporting üçün hazır yekun nəticə yaratmaqdır.

In [45]:
gold_kpi = pd.read_sql("""
SELECT
    COUNT(*) AS TOTAL_FLIGHTS,
    SUM(CASE WHEN FLIGHT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS COMPLETED_FLIGHTS,
    SUM(CANCELLED) AS CANCELLED_FLIGHTS,
    SUM(DIVERTED) AS DIVERTED_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
""", con=engine)

gold_kpi

,TOTAL_FLIGHTS,COMPLETED_FLIGHTS,CANCELLED_FLIGHTS,DIVERTED_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,999657,981515,15527,2615,18.6,4.39


#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış ümumi KPI nəticələri `gold.flight_kpi_summary` cədvəlinə yazılır.

In [46]:
gold_kpi.to_sql(
    "flight_kpi_summary",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

1

## 2. Aviaşirkətlər üzrə performans nəticələrinin hazırlanması

Bu mərhələdə aviaşirkətlər üzrə ümumi uçuş sayı, gecikmə faizi və orta çatma gecikməsi hesablanır. Məqsəd nəticəni Power BI üçün Gold layer-da hazırlamaqdır.

In [47]:
gold_airline = pd.read_sql("""
SELECT
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY AIRLINE_NAME
ORDER BY DELAY_RATE
""", con=engine)

gold_airline

,AIRLINE_NAME,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Hawaiian Airlines Inc.,13122,10.93,1.87
1,Alaska Airlines Inc.,29609,13.05,-1.06
2,Delta Air Lines Inc.,149858,13.65,0.28
3,American Airlines Inc.,124964,18.20,3.18
4,Skywest Airlines Inc.,100614,18.63,5.83
5,US Airways Inc.,34278,18.89,3.77
6,Southwest Airlines Co.,217546,19.04,4.39
7,Virgin America,10703,19.35,4.91
8,Atlantic Southeast Airlines,98165,19.64,6.66
9,United Air Lines Inc.,88596,20.54,5.33


#### Nəticənin Gold cədvəlinə yazılması

Aşağıdakı addımda hazırlanmış aviaşirkət performans nəticələri `gold.airline_performance` cədvəlinə yazılır.

In [48]:
gold_airline.to_sql(
    "airline_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

14

## 3. Aylıq performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların aylar üzrə ümumi sayı, gecikmə faizi, orta çatma gecikməsi, ləğv və diversion faizləri hesablanır.
Məqsəd nəticələri Power BI üçün hazır Gold cədvəlinə çevirməkdir.

In [49]:
gold_monthly = pd.read_sql("""
SELECT
    MONTH,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(100.0 * SUM(CANCELLED) / COUNT(*), 2) AS CANCELLATION_RATE,
    ROUND(100.0 * SUM(DIVERTED) / COUNT(*), 2) AS DIVERSION_RATE
FROM silver.flights_clean
GROUP BY MONTH
""", con=engine)

gold_monthly

,MONTH,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY,CANCELLATION_RATE,DIVERSION_RATE
0,April,83792,17.30,3.43,0.93,0.30
1,August,87623,18.60,4.51,1.01,0.30
2,December,82536,20.64,6.07,1.75,0.30
3,February,74041,23.16,8.28,4.75,0.23
4,January,80869,21.01,5.70,2.57,0.22
5,July,89974,20.67,6.21,0.95,0.29
6,June,86385,23.52,9.58,1.82,0.39
7,March,86649,19.41,5.01,2.15,0.23
8,May,85504,18.26,4.45,1.16,0.34
9,November,80361,15.17,0.88,1.00,0.22


#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış aylıq performans nəticələri `gold.monthly_performance` cədvəlinə yazılır.

In [50]:
gold_monthly.to_sql(
    "monthly_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

12

## 4. Günün vaxtına görə performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların günün müxtəlif vaxtlarına görə ümumi sayı, gecikmə faizi və orta çatma gecikməsi hesablanır.
Məqsəd hansı vaxt aralığında uçuş performansının daha zəif olduğunu Power BI-da göstərməkdir.

In [51]:
gold_departure_period = pd.read_sql("""
SELECT
    DEPARTURE_PERIOD,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY DEPARTURE_PERIOD
ORDER BY DELAY_RATE DESC
""", con=engine)

gold_departure_period

,DEPARTURE_PERIOD,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Evening,218254,25.34,9.02
1,Afternoon,368276,21.06,6.56
2,Morning,388979,13.09,0.21
3,Night,24148,9.09,-3.14


In [52]:
gold_departure_period.to_sql(
    "departure_period",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

4

## 5. Marşrutlar üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin və destination hava limanları üzrə uçuş sayı və gecikmə faizi hesablanır.
Məqsəd gecikmənin daha yüksək olduğu marşrutları Power BI üçün hazırlamaqdır.

In [53]:
gold_route = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
HAVING COUNT(*) >= 500
""", con=engine)

gold_route

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,TOTAL_FLIGHTS,DELAY_RATE
0,SEA,ANC,1201,19.93
1,BHM,ATL,609,11.92
2,AUS,ATL,561,11.91
3,CAE,ATL,510,13.81
4,CLT,ATL,1117,18.71
...,...,...,...,...
488,DFW,STL,507,16.02
489,SFO,SNA,808,18.18
490,ATL,TPA,1291,16.51
491,CLT,TPA,543,19.93


In [54]:
gold_route.to_sql(
    "route",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

493

## 6. Hava limanları üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin hava limanları üzrə orta taxi-out müddəti hesablanır.
Məqsəd yerüstü əməliyyatlarda daha çox vaxt itirilən hava limanlarını Power BI üçün hazırlamaqdır.

In [55]:
gold_airport = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(AVG(TAXI_OUT), 2) AS AVG_TAXI_OUT
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, ORIGIN_AIRPORT_NAME
""", con=engine)

gold_airport

,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,TOTAL_FLIGHTS,AVG_TAXI_OUT
0,ADK,Adak Airport,17,8.00
1,ABI,Abilene Regional Airport,453,9.20
2,ABR,Aberdeen Regional Airport,145,16.42
3,ATW,Appleton International Airport,535,15.69
4,OAJ,Albert J. Ellis Airport,217,14.18
...,...,...,...,...
316,CRW,Yeager Airport,454,13.66
317,SCE,University Park Airport,181,12.32
318,YUM,Yuma International Airport,351,14.10
319,HDN,Yampa Valley Airport (Yampa Valley Regional),115,18.21


In [56]:
gold_airport.to_sql(
    "airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

321

In [57]:
query = text("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'gold'
      AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME;
""")

with engine.connect() as connection:
    result = connection.execute(query)
    gold_tables = [row[0] for row in result]

print(f"Gold layer table count: {len(gold_tables)}")

for table in gold_tables:
    print(table)

Gold layer table count: 16
airline_performance
airport
delay_causes
delay_recovery
departure_period
dim_airline
dim_destination_airport
dim_month
dim_origin_airport
distance_impact
fact_flights
flight_kpi_summary
monthly_performance
route
weather_impact
weekday_weekend_performance


### 7. Həftəiçi və həftəsonu performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların həftəiçi və həftəsonu performansı müqayisə edilir.
Məqsəd gecikmə göstəricilərinin gün tipinə görə dəyişib-dəyişmədiyini Gold layer üçün hazırlamaqdır.

In [58]:
gold_weekday_weekend = pd.read_sql("""
SELECT
    IS_WEEKEND,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY IS_WEEKEND
""", con=engine)

gold_weekday_weekend

,IS_WEEKEND,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,0,738808,19.10,4.92
1,1,260849,17.16,2.88


#### Nəticənin Gold cədvəlinə yazılması

In [59]:
gold_weekday_weekend.to_sql(
    "weekday_weekend_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 8. Gecikmə səbəbləri üzrə nəticələrin hazırlanması

Bu mərhələdə gecikmiş uçuşlarda əsas gecikmə səbəblərinin orta təsiri hesablanır.
Məqsəd gecikmələrin daha çox hansı səbəblərdən yarandığını Gold layer üçün hazırlamaqdır.

In [60]:
gold_delay_causes = pd.read_sql("""
SELECT
    ROUND(AVG(AIRLINE_DELAY), 2) AS AIRLINE_DELAY,
    ROUND(AVG(LATE_AIRCRAFT_DELAY), 2) AS LATE_AIRCRAFT_DELAY,
    ROUND(AVG(AIR_SYSTEM_DELAY), 2) AS AIR_SYSTEM_DELAY,
    ROUND(AVG(WEATHER_DELAY), 2) AS WEATHER_DELAY,
    ROUND(AVG(SECURITY_DELAY), 2) AS SECURITY_DELAY
FROM silver.flights_clean
WHERE IS_DELAYED = 1
""", con=engine)

gold_delay_causes

,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,AIR_SYSTEM_DELAY,WEATHER_DELAY,SECURITY_DELAY
0,18.92,23.38,13.49,2.97,0.07


#### Nəticənin Gold cədvəlinə yazılması

In [61]:
gold_delay_causes.to_sql(
    "delay_causes",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

1

### 9. Hava şəraitinin gecikmələrə təsirinin hazırlanması

Bu mərhələdə yağışlı və yağışsız hava şəraitində uçuş gecikmələri müqayisə edilir.
Məqsəd hava şəraitinin gecikmə göstəricilərinə təsirini Gold layer üçün hazırlamaqdır.

In [62]:
gold_weather_impact = pd.read_sql("""
SELECT
    CASE 
        WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain'
        ELSE 'No Rain'
    END AS WEATHER_GROUP,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
WHERE WEATHER_PRECIPITATION IS NOT NULL
GROUP BY 
    CASE 
        WHEN WEATHER_PRECIPITATION > 0 THEN 'Rain'
        ELSE 'No Rain'
    END
""", con=engine)

gold_weather_impact

,WEATHER_GROUP,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Rain,27.82,13.42
1,No Rain,17.36,3.17


#### Nəticənin Gold cədvəlinə yazılması

In [63]:
gold_weather_impact.to_sql(
    "weather_impact",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 10. Məsafənin gecikmələrə təsirinin hazırlanması

Bu mərhələdə uzun məsafəli və digər uçuşların gecikmə göstəriciləri müqayisə edilir.
Məqsəd uçuş məsafəsinin gecikmə performansına təsirini Gold layer üçün hazırlamaqdır.

In [64]:
gold_distance_impact = pd.read_sql("""
SELECT
    IS_LONG_DISTANCE,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(AIR_TIME), 2) AS AVG_AIR_TIME
FROM silver.flights_clean
GROUP BY IS_LONG_DISTANCE
""", con=engine)

gold_distance_impact

,IS_LONG_DISTANCE,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY,AVG_AIR_TIME
0,0,864207,18.62,4.79,90.97
1,1,135450,18.43,1.84,256.54


#### Nəticənin Gold cədvəlinə yazılması

In [65]:
gold_distance_impact.to_sql(
    "distance_impact",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

2

### 11. Aviaşirkətlərin gecikməni bərpa etmə performansının hazırlanması

Bu mərhələdə aviaşirkətlərin uçuş zamanı başlanğıc gecikmənin nə qədərini bərpa edə bildiyi hesablanır.
Məqsəd əməliyyat səmərəliliyini Gold layer üçün hazırlamaqdır.


In [66]:
gold_delay_recovery = pd.read_sql("""
SELECT
    AIRLINE_NAME,
    ROUND(AVG(DEPARTURE_DELAY), 2) AS AVG_DEPARTURE_DELAY,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(AVG(DEPARTURE_DELAY - ARRIVAL_DELAY), 2) AS TIME_RECOVERED
FROM silver.flights_clean
WHERE DEPARTURE_DELAY > 0
  AND ARRIVAL_DELAY IS NOT NULL
GROUP BY AIRLINE_NAME
ORDER BY TIME_RECOVERED DESC
""", con=engine)

gold_delay_recovery

,AIRLINE_NAME,AVG_DEPARTURE_DELAY,AVG_ARRIVAL_DELAY,TIME_RECOVERED
0,United Air Lines Inc.,32.38,23.52,8.86
1,Delta Air Lines Inc.,29.89,22.99,6.90
2,Southwest Airlines Co.,26.87,21.20,5.67
3,American Airlines Inc.,33.50,28.45,5.05
4,JetBlue Airways,37.20,32.99,4.21
5,Virgin America,29.72,25.59,4.13
6,Alaska Airlines Inc.,25.86,22.39,3.47
7,Skywest Airlines Inc.,38.94,36.77,2.18
8,US Airways Inc.,28.49,26.46,2.03
9,Atlantic Southeast Airlines,41.17,39.62,1.54


#### Nəticənin Gold cədvəlinə yazılması

In [67]:
gold_delay_recovery.to_sql(
    "delay_recovery",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

14

## Power BI üçün Star Schema modelinin hazırlanması

In [68]:
silver_structure = pd.read_sql("""
SELECT
    COLUMN_NAME,
    DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver'
  AND TABLE_NAME = 'flights_clean'
ORDER BY ORDINAL_POSITION
""", con=engine)

silver_structure

,COLUMN_NAME,DATA_TYPE
0,MONTH,varchar
1,DAY,smallint
2,DAY_OF_WEEK,smallint
3,AIRLINE,varchar
4,ORIGIN_AIRPORT,varchar
5,DESTINATION_AIRPORT,varchar
6,DISTANCE,int
7,SCHEDULED_DEPARTURE,time
8,SCHEDULED_ARRIVAL,time
9,DEPARTURE_DELAY,real


### 12. Airline Dimension cədvəlinin hazırlanması

Bu mərhələdə aviaşirkətlər üçün unikal dimension cədvəli hazırlanır.
Dim Airline cədvəli Fact Flights ilə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [69]:
dim_airline = pd.read_sql("""
SELECT DISTINCT
    AIRLINE_NAME,
    AIRLINE
FROM silver.flights_clean
WHERE AIRLINE_NAME IS NOT NULL
ORDER BY AIRLINE_NAME
""", con=engine)


dim_airline.insert(
    0,
    "AIRLINE_ID",
    range(1, len(dim_airline) + 1)
)

dim_airline

,AIRLINE_ID,AIRLINE_NAME,AIRLINE
0,1,Alaska Airlines Inc.,AS
1,2,American Airlines Inc.,AA
2,3,American Eagle Airlines Inc.,MQ
3,4,Atlantic Southeast Airlines,EV
4,5,Delta Air Lines Inc.,DL
5,6,Frontier Airlines Inc.,F9
6,7,Hawaiian Airlines Inc.,HA
7,8,JetBlue Airways,B6
8,9,Skywest Airlines Inc.,OO
9,10,Southwest Airlines Co.,WN


#### Dim Airline cədvəlinin Gold layer-ə yazılması

In [70]:
dim_airline.to_sql(
    "dim_airline",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

14

### 13. Origin Airport Dimension cədvəlinin hazırlanması

Bu mərhələdə uçuşların başladığı hava limanları üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə origin airport üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [71]:
dim_origin_airport = pd.read_sql("""
SELECT DISTINCT
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME,
    ORIGIN_STATE
FROM silver.flights_clean
WHERE ORIGIN_AIRPORT IS NOT NULL
ORDER BY ORIGIN_AIRPORT
""", con=engine)

dim_origin_airport.insert(
    0,
    "ORIGIN_AIRPORT_ID",
    range(1, len(dim_origin_airport) + 1)
)

dim_origin_airport

,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_STATE
0,1,ABE,Lehigh Valley International Airport,PA
1,2,ABI,Abilene Regional Airport,TX
2,3,ABQ,Albuquerque International Sunport,NM
3,4,ABR,Aberdeen Regional Airport,SD
4,5,ABY,Southwest Georgia Regional Airport,GA
...,...,...,...,...
316,317,WRG,Wrangell Airport,AK
317,318,WYS,Westerly State Airport,MT
318,319,XNA,Northwest Arkansas Regional Airport,AR
319,320,YAK,Yakutat Airport,AK


#### Origin Airport Dimension cədvəlinin Gold layer-ə yazılması

In [72]:
dim_origin_airport.to_sql(
    "dim_origin_airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

321

### 14. Destination Airport Dimension cədvəlinin hazırlanması

Bu mərhələdə uçuşların çatdığı hava limanları üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə destination airport üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [73]:
dim_destination_airport = pd.read_sql("""
SELECT DISTINCT
    DESTINATION_AIRPORT,
    DESTINATION_AIRPORT_NAME
FROM silver.flights_clean
WHERE DESTINATION_AIRPORT IS NOT NULL
ORDER BY DESTINATION_AIRPORT
""", con=engine)

dim_destination_airport.insert(
    0,
    "DESTINATION_AIRPORT_ID",
    range(1, len(dim_destination_airport) + 1)
)

dim_destination_airport

,DESTINATION_AIRPORT_ID,DESTINATION_AIRPORT,DESTINATION_AIRPORT_NAME
0,1,ABE,Lehigh Valley International Airport
1,2,ABI,Abilene Regional Airport
2,3,ABQ,Albuquerque International Sunport
3,4,ABR,Aberdeen Regional Airport
4,5,ABY,Southwest Georgia Regional Airport
...,...,...,...
316,317,WRG,Wrangell Airport
317,318,WYS,Westerly State Airport
318,319,XNA,Northwest Arkansas Regional Airport
319,320,YAK,Yakutat Airport


#### Destination Airport Dimension cədvəlinin Gold layer-ə yazılması

In [74]:
dim_destination_airport.to_sql(
    "dim_destination_airport",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

321

### 15. Month Dimension cədvəlinin hazırlanması

Bu mərhələdə aylar üçün unikal dimension cədvəli hazırlanır.
Bu cədvəl Fact Flights ilə ay üzrə one-to-many əlaqə qurmaq üçün istifadə olunacaq.

In [75]:
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

In [76]:
dim_month = pd.read_sql("""
SELECT DISTINCT
    MONTH
FROM silver.flights_clean
WHERE MONTH IS NOT NULL
ORDER BY MONTH
""", con=engine)

dim_month.insert(0, "MONTH_ID", dim_month["MONTH"].map(month_map))
dim_month = dim_month.sort_values("MONTH_ID").reset_index(drop=True)

dim_month

,MONTH_ID,MONTH
0,1,January
1,2,February
2,3,March
3,4,April
4,5,May
5,6,June
6,7,July
7,8,August
8,9,September
9,10,October


#### Month Dimension cədvəlinin Gold layer-ə yazılması

In [77]:
dim_month.to_sql(
    "dim_month",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

12

### 16. Fact Flights cədvəlinin hazırlanması

Bu mərhələdə hər uçuşu təmsil edən əsas Fact cədvəli hazırlanır.
Dimension cədvəllərinin ID-ləri Fact Flights cədvəlinə əlavə edilir və Power BI-də one-to-many əlaqələrin qurulması üçün istifadə olunur.

In [78]:
fact_flights = pd.read_sql("""
SELECT
    a.AIRLINE_ID,
    a.AIRLINE_NAME,
    a.AIRLINE,
    o.ORIGIN_AIRPORT,
    o.ORIGIN_AIRPORT_NAME,
    o.ORIGIN_STATE,
    d.DESTINATION_AIRPORT_ID,
    d.DESTINATION_AIRPORT,
    d.DESTINATION_AIRPORT_NAME,
    m.MONTH_ID,
    f.IS_WEEKEND,
    f.DEPARTURE_PERIOD,
    f.DISTANCE,
    f.IS_LONG_DISTANCE,
    f.DEPARTURE_DELAY,
    f.ARRIVAL_DELAY,
    f.TAXI_OUT,
    f.AIR_TIME,
    f.CANCELLED,
    f.DIVERTED,
    f.IS_DELAYED,
    f.FLIGHT_STATUS,
    f.AIRLINE_DELAY,
    f.LATE_AIRCRAFT_DELAY,
    f.AIR_SYSTEM_DELAY,
    f.WEATHER_DELAY,
    f.SECURITY_DELAY,
    f.WEATHER_PRECIPITATION

FROM silver.flights_clean f

LEFT JOIN gold.dim_airline a
    ON f.AIRLINE_NAME = a.AIRLINE_NAME

LEFT JOIN gold.dim_origin_airport o
    ON f.ORIGIN_AIRPORT = o.ORIGIN_AIRPORT

LEFT JOIN gold.dim_destination_airport d
    ON f.DESTINATION_AIRPORT = d.DESTINATION_AIRPORT

LEFT JOIN gold.dim_month m
    ON f.MONTH = m.MONTH
""", con=engine)

fact_flights

,AIRLINE_ID,AIRLINE_NAME,AIRLINE,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_STATE,DESTINATION_AIRPORT_ID,DESTINATION_AIRPORT,DESTINATION_AIRPORT_NAME,MONTH_ID,...,CANCELLED,DIVERTED,IS_DELAYED,FLIGHT_STATUS,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,AIR_SYSTEM_DELAY,WEATHER_DELAY,SECURITY_DELAY,WEATHER_PRECIPITATION
0,8,JetBlue Airways,B6,SJU,Luis Muñoz Marín International Airport,PR,115,FLL,Fort Lauderdale-Hollywood International Airport,10,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
1,5,Delta Air Lines Inc.,DL,ATL,Hartsfield-Jackson Atlanta International Airport,GA,195,MDW,Chicago Midway International Airport,1,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
2,10,Southwest Airlines Co.,WN,ATL,Hartsfield-Jackson Atlanta International Airport,GA,176,LAX,Los Angeles International Airport,8,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
3,2,American Airlines Inc.,AA,CLT,Charlotte Douglas International Airport,NC,40,BOS,Gen. Edward Lawrence Logan International Airport,12,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
4,10,Southwest Airlines Co.,WN,LAX,Los Angeles International Airport,CA,195,MDW,Chicago Midway International Airport,7,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999652,9,Skywest Airlines Inc.,OO,HLN,Helena Regional Airport,MT,86,DEN,Denver International Airport,7,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
999653,9,Skywest Airlines Inc.,OO,PHX,Phoenix Sky Harbor International Airport,AZ,92,DRO,Durango-La Plata County Airport,9,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
999654,5,Delta Air Lines Inc.,DL,LGA,LaGuardia Airport (Marine Air Terminal),NY,115,FLL,Fort Lauderdale-Hollywood International Airport,4,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0
999655,2,American Airlines Inc.,AA,PHL,Philadelphia International Airport,PA,203,MIA,Miami International Airport,4,...,0,0,0.0,Completed,NaN,NaN,NaN,NaN,NaN,0.0


#### Fact Flights cədvəlinin Gold layer-ə yazılması

In [79]:
fact_flights.to_sql(
    "fact_flights",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

65

### 17. Star Schema cədvəllərinin yoxlanılması

In [80]:
pd.read_sql("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'gold'
  AND TABLE_NAME IN (
      'dim_airline',
      'dim_origin_airport',
      'dim_destination_airport',
      'dim_month',
      'fact_flights'
  )
ORDER BY TABLE_NAME
""", con=engine)

,TABLE_NAME
0,dim_airline
1,dim_destination_airport
2,dim_month
3,dim_origin_airport
4,fact_flights


In [81]:
query = text("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'gold'
      AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME;
""")

with engine.connect() as connection:
    result = connection.execute(query)
    gold_tables = [row[0] for row in result]

print(f"Gold layer table count: {len(gold_tables)}")

for table in gold_tables:
    print(table)

Gold layer table count: 16
airline_performance
airport
delay_causes
delay_recovery
departure_period
dim_airline
dim_destination_airport
dim_month
dim_origin_airport
distance_impact
fact_flights
flight_kpi_summary
monthly_performance
route
weather_impact
weekday_weekend_performance
